In [1]:
import cv2
import json
import numpy as np

In [2]:
# Load First Video Frame
# done drawing polygon
#VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_1/CAM_3-entry.mp4"  
#VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_1/CAM_2-zone.mp4"
#VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_1/CAM_1-zone.mp4"
#VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_1/CAM_5-billing.mp4"
# current
VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_2/entry_1.mp4"
# remains
#VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_2/entry_2.mp4"
#VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_2/billing_area.mp4"
#VIDEO_PATH = "D:/store-intelligence-system/data/raw/Store_2/zone.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

ret, frame = cap.read()

cap.release()

if not ret:
    raise ValueError("Unable to read frame")


display_frame = frame.copy()

print(display_frame.shape)

(1080, 960, 3)


In [3]:
# Resize Frame For Easier Annotation
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

frame = cv2.resize(
    frame,
    (FRAME_WIDTH, FRAME_HEIGHT)
)

base_frame = frame.copy()

display_frame = frame.copy()

In [4]:
# GLOBAL VARIABLES
current_points = [] 
zones = {} 
zone_counter = 1

In [5]:
# MOUSE CALLBACK
def mouse_callback(event, x, y, flags, param):

    global current_points
    global display_frame

    if event == cv2.EVENT_LBUTTONDOWN:

        current_points.append((x, y))

        print(f"Point Added: {(x, y)}")

        # Draw point
        cv2.circle(
            display_frame,
            (x, y),
            5,
            (0, 255, 0),
            -1
        )

        # Draw connecting line
        if len(current_points) > 1:

            cv2.line(
                display_frame,
                current_points[-2],
                current_points[-1],
                (255, 0, 0),
                2
            )


In [6]:
# WINDOW SETUP
cv2.namedWindow(
    "Polygon Editor",
    cv2.WINDOW_NORMAL
)

cv2.resizeWindow(
    "Polygon Editor",
    FRAME_WIDTH,
    FRAME_HEIGHT
)

cv2.setMouseCallback(
    "Polygon Editor",
    mouse_callback
)

In [7]:
# Create Window
cv2.namedWindow("Polygon Editor")

cv2.setMouseCallback(
    "Polygon Editor",
    mouse_callback
)

In [8]:
# ============================================================
# MAIN LOOP
# ============================================================

while True:

    cv2.imshow(
        "Polygon Editor",
        display_frame
    )

    key = cv2.waitKey(1) & 0xFF


    # ========================================================
    # QUIT
    # ========================================================

    if key == ord("q"):

        print("Exiting editor")

        break


    # ========================================================
    # CLOSE POLYGON
    # ========================================================

    elif key == ord("c"):

        if len(current_points) > 2:

            polygon = np.array(
                current_points,
                dtype=np.int32
            )

            # Draw closed polygon
            cv2.polylines(
                display_frame,
                [polygon],
                isClosed=True,
                color=(255, 0, 0),
                thickness=2
            )

            # Refresh frame first
            cv2.imshow(
                "Polygon Editor",
                display_frame
            )

            cv2.waitKey(1)

            # Ask zone name AFTER rendering
            zone_name = input(
                "\nEnter Zone Name: "
            )

            # Save polygon
            zones[zone_name] = (
                current_points.copy()
            )

            print(
                f"Saved: {zone_name}"
            )

            # Draw label
            first_point = current_points[0]

            cv2.putText(
                display_frame,
                zone_name,
                first_point,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

            current_points.clear()


    # ========================================================
    # SAVE JSON
    # ========================================================

    elif key == ord("s"):

        with open(
            "D:/store-intelligence-system/configs/store_2/entry_1_zone.json",
            "w"
        ) as f:

            json.dump(
                zones,
                f,
                indent=4
            )

        print(
            "Zones saved successfully"
        )


    # ========================================================
    # RESET CURRENT POLYGON
    # ========================================================

    elif key == ord("r"):

        print("Resetting current polygon")

        current_points.clear()

        # Restore original frame
        display_frame = base_frame.copy()

        # Re-draw all saved polygons
        for zone_name, points in zones.items():

            polygon = np.array(
                points,
                dtype=np.int32
            )

            cv2.polylines(
                display_frame,
                [polygon],
                isClosed=True,
                color=(255, 0, 0),
                thickness=2
            )

            # Draw zone label
            first_point = points[0]

            cv2.putText(
                display_frame,
                zone_name,
                first_point,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )


# ============================================================
# CLEANUP
# ============================================================

cv2.destroyAllWindows()

Point Added: (410, 448)
Point Added: (405, 522)
Point Added: (918, 470)
Point Added: (893, 414)
Saved: entry_zone
Point Added: (403, 559)
Point Added: (398, 714)
Point Added: (983, 717)
Resetting current polygon
Point Added: (403, 581)
Point Added: (404, 719)
Point Added: (970, 711)
Point Added: (941, 528)
Saved: entrance_buffer_zone
Point Added: (70, 251)
Point Added: (27, 420)
Point Added: (1228, 336)
Point Added: (1186, 193)
Saved: outside_walkway_zone
Zones saved successfully
Exiting editor


In [ ]:
# ============================================================
# FINAL OUTPUT
# ============================================================

print("\nFinal Zones:\n")

print(
    json.dumps(
        zones,
        indent=4
    )
)


Final Zones:

{
    "entry_zone": [
        [
            731,
            355
        ],
        [
            528,
            240
        ],
        [
            716,
            113
        ],
        [
            733,
            5
        ],
        [
            936,
            4
        ],
        [
            883,
            175
        ]
    ],
    "outside_walkway_zone": [
        [
            687,
            715
        ],
        [
            1053,
            176
        ],
        [
            1276,
            221
        ],
        [
            1275,
            713
        ]
    ],
    "promo_zone": [
        [
            190,
            320
        ],
        [
            466,
            157
        ],
        [
            479,
            530
        ],
        [
            307,
            702
        ]
    ]
}
